# QLattice — dynamic operational margins

Symbolic regression where loading is read from **(Q_pred − Q_meas)** vs **flow-dependent margins** (same code path as `src/qlattice_pipeline.py`).

- **`Helm.from_dynamic_bounds_csv`** — filtered CSV, stratified split, labels Loaded=1 / Near L.U=0 / Unloaded=-1; includes **`g (m/s2)`**.
- **`QLatticeWrapper.fit_dynamic_bounds`** — Feyn fit in this framing (not direct classification `fit(X, loading)`).
- **`Helm.evolv_dynamic_bounds_qlattice`** — hyperparameter grid like `evolv_model`, scoring **test** dynamic-bounds metrics (`f1_lu` by default).

Default CSV: `default_qlattice_csv_path()` (repo `datasets/processed_well_data.csv`). The **last code cell** saves **`notebook_exports/*.csv`** (same schema as the CLI) and shows **Turner + residual plots**.

CLI: `%run qlattice_standalone.py --plot-dir notebook_exports --show-plots ...` or `run_qlattice_from_argv([...])`.

In [ ]:
%matplotlib inline

import numpy as np
from pathlib import Path

from src.utils import Helm, QLatticeWrapper, default_qlattice_csv_path
from src.qlattice_pipeline import TURNER_FEATURE_COLS

SEED = 42
data_path = default_qlattice_csv_path()
D = Helm.from_dynamic_bounds_csv(data_path, seed=SEED, test_size=0.2)
print("Features:", TURNER_FEATURE_COLS)
print("Train / test:", len(D.gsflow_train), len(D.gsflow_test))

NOTEBOOK_EXPORT = Path("notebook_exports")
NOTEBOOK_EXPORT.mkdir(parents=True, exist_ok=True)

## Hyperparameter grid (`evolv_dynamic_bounds_qlattice`)

Keys match `QLatticeWrapper`: `max_complexity`, `n_epochs`, `select_by`, `loss`, `val_frac`, `top_models`, `qcr_proxy_margin_scale`, `linear_gas`, `no_val_select`, `qcr_use_measured_gas`.

`mode`: **`qcr_labels`**, **`gas`**, or optional **`qcr_custom`** with `--qcr-npz`.

In [ ]:
hparam_grid = {
    "max_complexity": [10, 14],
    "n_epochs": [32, 48],
    "select_by": ["hybrid"],
    "qcr_proxy_margin_scale": [2.5],
}

best = D.evolv_dynamic_bounds_qlattice(
    hparam_grid,
    mode="qcr_labels",
    score_by="f1_lu",
)

print("Test metrics:", best.dynamic_bounds_metrics)
print(best.express())

## Single fit (no grid)

Call **`fit_dynamic_bounds`** when hyperparameters are fixed.

In [ ]:
m = QLatticeWrapper(
    mode="qcr_labels",
    seed=SEED,
    max_complexity=12,
    n_epochs=48,
    select_by="hybrid",
    qcr_proxy_margin_scale=2.5,
)
m.fit_dynamic_bounds(
    D.train_X_df,
    D.test_X_df,
    D.gsflow_train,
    D.gsflow_test,
    D.loading_train,
    D.loading_test,
)
print(m.dynamic_bounds_metrics)
print(m.express())

## Save CSV + Turner plots (notebook)

Writes the same columns as the standalone pipeline (`Q_meas_m3day`, `Q_cr_pred_m3day`, `Test status`, δ refs, …). Runs **`export_qlattice_run(m, ...)`** for the single-fit model and **`export_qlattice_run(best, ...)`** for the grid search model when those variables exist.

In [ ]:
from src.qlattice_pipeline import (
    eval_turner_train_test_metrics,
    plot_residual_qminus_qmeas,
    plot_turner_qmeas_vs_qrate,
    save_pipeline_predictions_csv,
)


def export_qlattice_run(model: QLatticeWrapper, tag: str) -> None:
    if getattr(model, "pred_train", None) is None:
        print(f"Skip {tag}: no predictions on model (run fit_dynamic_bounds first).")
        return
    ev = eval_turner_train_test_metrics(
        model.pred_train,
        D.gsflow_train,
        D.loading_train,
        model.pred_test,
        D.gsflow_test,
        D.loading_test,
    )
    q_meas = np.concatenate([D.gsflow_train, D.gsflow_test])
    q_pred = np.concatenate([model.pred_train, model.pred_test])
    is_train = np.concatenate(
        [np.ones(len(D.gsflow_train), dtype=bool), np.zeros(len(D.gsflow_test), dtype=bool)]
    )
    load_all = np.concatenate([D.loading_train, D.loading_test])
    save_pipeline_predictions_csv(
        NOTEBOOK_EXPORT,
        pipeline="qlattice",
        mode=tag,
        seed=SEED,
        q_meas=q_meas,
        q_pred=q_pred,
        loading=load_all,
        is_train=is_train,
        pred_column="Q_cr_pred_m3day",
        delta_pos=float(ev["dpos"]),
        delta_neg=float(ev["dneg"]),
        source_csv=str(data_path),
    )
    base = NOTEBOOK_EXPORT / f"qlattice_nb_{tag}_turner"
    plot_turner_qmeas_vs_qrate(
        q_meas,
        q_pred,
        is_train,
        ev["dpos"],
        ev["dneg"],
        f"QLattice Turner plot ({tag})",
        r"Symbolic $Q_{\mathrm{cr}}$ (m³/day)",
        base,
        loading=load_all,
        show=True,
    )
    plot_residual_qminus_qmeas(
        q_meas,
        q_pred,
        is_train,
        ev["dpos"],
        ev["dneg"],
        f"QLattice residual ({tag})",
        base,
        loading=load_all,
        show=True,
    )


try:
    export_qlattice_run(best, "qcr_labels_grid")
except NameError:
    print("Grid model `best` not defined — run the hyperparameter cell first.")
try:
    export_qlattice_run(m, "qcr_labels_single")
except NameError:
    print("Single model `m` not defined — run the single-fit cell first.")